# Sufficiency Probe Training

This notebook is Phase 4 of the project: it trains a small probe to predict
"is this a safe place to stop generating?" directly from the model's hidden
state, and compares it against the entropy baseline already measured on the
Raspberry Pi (precision ~0.96-0.97, recall 0.31 -> 1.00 across a threshold
sweep -- see `qa_e1_report.txt` and Section 6.3 of the paper).

**What this notebook needs from you:** the file `qa_gen_run3.jsonl` from
the Pi (`~/e1/out/qa_gen_run3.jsonl`). Upload it using the cell below.

**What this notebook produces:** a trained probe, a precision/recall table
and plot in the same format as the entropy baseline, and a saved probe file
you can bring back to the Pi later.

Runtime: GPU (Runtime -> Change runtime type -> T4 GPU). Takes about
10-15 minutes end to end on the free tier.

## 1. Setup

In [ ]:
!pip install -q transformers accelerate scikit-learn matplotlib

In [ ]:
import json, math, re, random
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU")

## 2. Upload your data

Upload `qa_gen_run3.jsonl` from the Pi (`~/e1/out/qa_gen_run3.jsonl`).
This is the file with the model's full responses to the 304 QA questions.

In [ ]:
from google.colab import files
uploaded = files.upload()
gen_path = list(uploaded.keys())[0]
print("using:", gen_path)

## 3. Load the model

Same model used on the Pi: Llama-3.2-1B-Instruct. Loaded here in full
precision (not the Q4 GGUF quantization used on the Pi) because we need
access to hidden states, which the quantized llama.cpp build doesn't
expose. This is fine -- the probe only needs to learn a *relative* pattern
in the hidden states, and the model's behavior (which tokens it generates)
was already captured on the Pi and is not being reproduced here.

In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
# This is an ungated mirror of meta-llama/Llama-3.2-1B-Instruct, used by
# default to avoid HuggingFace login friction. If you have a HF token and
# prefer the original repo, swap to:
# MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, output_hidden_states=True
).to(device)
model.eval()
print("model loaded, hidden size:", model.config.hidden_size)

## 4. Rebuild boundaries and labels from the raw responses

We recompute sentence boundaries and safe-stop labels directly from the
raw text and answer keys, using the same logic as the Pi's labeler
(`scripts/analysis/label_qa_boundaries.py`), so this notebook is
self-contained and doesn't depend on a separate labels file.

In [ ]:
BOUNDARY = re.compile(r"(?<=[.!?\n])\s+")

def contains_answer(text, answers):
    t = text.lower()
    return any(a.lower() in t for a in answers)

def boundaries(text):
    offs = [m.start() for m in BOUNDARY.finditer(text)]
    if not offs or offs[-1] < len(text):
        offs.append(len(text))
    return offs

SYSTEM = ("You are a helpful assistant. Answer the user's question clearly "
          "and correctly.")

def chat_prompt(question):
    return ("<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            + SYSTEM + "<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            + question + "<|eot_id|><|start_header_id|>assistant"
            "<|end_header_id|>\n\n")

records = [json.loads(l) for l in open(gen_path)]
print(f"loaded {len(records)} responses")

# Build (prompt_prefix, boundary_char_offset, safe_stop_label) examples.
examples = []
for r in records:
    text, answers = r["text"], r["answers"]
    full_ok = contains_answer(text, answers)
    if not full_ok:
        continue  # only label responses the model got right end-to-end,
                  # same rule as the Pi labeler
    offs = boundaries(text)
    for b_idx, off in enumerate(offs):
        trunc = text[:off]
        safe = contains_answer(trunc, answers)
        examples.append({
            "id": r["id"], "question": r["question"],
            "prefix_text": trunc, "boundary": b_idx, "safe_stop": safe,
        })

n_safe = sum(1 for e in examples if e["safe_stop"])
print(f"{len(examples)} boundary examples, {n_safe} labeled safe-to-stop "
      f"({n_safe/len(examples)*100:.1f}%)")

## 5. Extract hidden states at each boundary

For each labeled boundary, we run the model on the prompt up through that
point and take the hidden state of the **last token**, from the model's
**final layer**. That single vector is what the probe learns from -- it is
the model's internal summary of "everything generated so far," and the
hypothesis is that this vector looks different when the answer is already
complete versus when it is not.

This is the slow part (one forward pass per boundary, ~450 boundaries).
Expect a few minutes on a T4 GPU.

In [ ]:
@torch.no_grad()
def get_hidden_state(prompt_prefix):
    full_text = chat_prompt(examples[0]["question"])  # placeholder, overwritten below
    inputs = tokenizer(full_text, return_tensors="pt").to(device)
    out = model(**inputs, output_hidden_states=True)
    # last layer, last token position
    return out.hidden_states[-1][0, -1, :].float().cpu().numpy()

@torch.no_grad()
def get_hidden_state_for(question, prefix_text):
    full_text = chat_prompt(question) + prefix_text
    inputs = tokenizer(full_text, return_tensors="pt", truncation=True,
                       max_length=1024).to(device)
    out = model(**inputs, output_hidden_states=True)
    return out.hidden_states[-1][0, -1, :].float().cpu().numpy()

X, y, meta = [], [], []
for i, ex in enumerate(examples):
    h = get_hidden_state_for(ex["question"], ex["prefix_text"])
    X.append(h)
    y.append(1 if ex["safe_stop"] else 0)
    meta.append(ex)
    if i % 50 == 0:
        print(f"{i+1}/{len(examples)} boundaries processed")

X = np.array(X)
y = np.array(y)
print("feature matrix:", X.shape, "label vector:", y.shape)

## 6. Train the probe

Two things are handled carefully here, both found necessary during
earlier runs of this notebook:

1. **Grouping by response, not by boundary, when splitting train/test.**
   A single response can produce several boundaries that share almost the
   same text and hidden state. If boundaries from the same response end
   up on both sides of the split, the probe can partly just recognize
   "I have seen this response before" rather than learning a general
   pattern, which inflates test performance artificially.
2. **Keeping only boundaries from multi-boundary responses for
   evaluation.** Most responses in this dataset are a single sentence,
   which is trivially "safe to stop" -- there is no real decision to
   make. Including these would make the test set overwhelmingly one
   class and the resulting precision/recall numbers close to
   meaningless. We restrict evaluation to responses with two or more
   boundaries, which is where a genuine stop-or-continue choice exists,
   matching the same filtering used for the entropy baseline on the Pi.

A small MLP (well under 1M parameters relative to the LLM itself) is
trained on the resulting hidden-state vectors to predict whether
stopping at a given boundary is safe.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, average_precision_score
from collections import defaultdict
import random

# --- Step A: group by response id, split by response (not by boundary) ---
unique_ids = list(set(e["id"] for e in examples))
random.seed(42)
random.shuffle(unique_ids)
n_test_ids = int(len(unique_ids) * 0.25)
test_ids = set(unique_ids[:n_test_ids])

train_idx = [i for i, e in enumerate(examples) if e["id"] not in test_ids]
test_idx  = [i for i, e in enumerate(examples) if e["id"] in test_ids]

# --- Step B: keep only genuinely hard (multi-boundary) examples ---
def filter_hard_boundaries(idx_list, examples_list):
    by_id = defaultdict(list)
    for i in idx_list:
        by_id[examples_list[i]["id"]].append(i)
    kept = []
    for rid, idxs in by_id.items():
        if len(idxs) >= 2:
            kept.extend(idxs)
    return kept

train_idx_hard = filter_hard_boundaries(train_idx, examples)
test_idx_hard = filter_hard_boundaries(test_idx, examples)

X_train, y_train = X[train_idx_hard], y[train_idx_hard]
X_test, y_test = X[test_idx_hard], y[test_idx_hard]

print(f"train (hard only): {len(X_train)} boundaries")
print(f"test (hard only): {len(X_test)} boundaries")
print(f"test set positives: {sum(y_test)}/{len(y_test)}  "
      f"({'CAUTION: very few negatives, expect noisy estimates' if (len(y_test)-sum(y_test)) < 8 else 'ok'})")

assert X_test.shape[0] == y_test.shape[0], "X_test/y_test mismatch -- rerun this cell fully"

# --- Step C: scale + train the probe ---
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

probe = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation="relu", max_iter=500, random_state=42,
    early_stopping=True, validation_fraction=0.15,
)
probe.fit(X_train_s, y_train)

n_params = (X.shape[1] * 64 + 64) + (64 * 1 + 1)
print(f"probe parameters: {n_params:,} (target: < 1,000,000)")

## 7. Evaluate: precision/recall, compared against the entropy baseline

The entropy numbers below are hardcoded from the Pi measurement
(`qa_e1_report.txt`, Section 6.3 of the paper) so the two curves can be
plotted on the same axes. The question this answers: does the learned
probe sit *above* the entropy curve (better recall at the same precision,
or better precision at the same recall), or does it just match it?

In [ ]:
assert X_test_s.shape[0] == y_test.shape[0], (
    f"Mismatch: X_test_s has {X_test_s.shape[0]} rows, y_test has "
    f"{y_test.shape[0]}. Re-run the previous cell (Section 6) fully before "
    f"this one -- do not run cells out of order."
)

probs = probe.predict_proba(X_test_s)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, probs)
ap = average_precision_score(y_test, probs)
print(f"probe average precision: {ap:.3f}")

entropy_recall =    [0.308, 0.373, 0.451, 0.535, 0.655, 0.794, 0.961, 1.000, 1.000]
entropy_precision = [0.957, 0.964, 0.961, 0.963, 0.963, 0.963, 0.965, 0.966, 0.966]

print("\nProbe operating points (subsample):")
print(f"{'threshold':>10} {'precision':>10} {'recall':>8}")
step = max(1, len(thresholds) // 10)
for i in range(0, len(thresholds), step):
    print(f"{thresholds[i]:>10.3f} {precisions[i]:>10.3f} {recalls[i]:>8.3f}")

n_neg = len(y_test) - sum(y_test)
print(f"\ntest set: {len(y_test)} examples, {sum(y_test)} positive, {n_neg} negative")
if n_neg < 8:
    print("CAUTION: very few negative examples in the test set -- "
          "precision/recall estimates have wide uncertainty. Treat this "
          "as a pilot result, not a final claim.")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 10, "font.family": "serif",
                     "axes.spines.top": False, "axes.spines.right": False})

fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.plot(recalls, precisions, color="#4f81c4", linewidth=1.8, label="Probe (S1)")
ax.plot(entropy_recall, entropy_precision, color="#c46b4f", linewidth=1.5,
       marker=".", linestyle="--", label="Entropy (S3)")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Learned probe vs. entropy baseline")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(fontsize=8, loc="lower left")
fig.tight_layout()
fig.savefig("fig_probe_vs_entropy.pdf")
fig.savefig("fig_probe_vs_entropy.png", dpi=150)
plt.show()
print("saved fig_probe_vs_entropy.pdf/.png")

## 8. Interpreting the result

Before reading too much into a single run, check the honest caveats:

- **Sample size.** With only ~450 boundary examples (and a minority
  labeled "unsafe"), this is a small dataset for training any classifier.
  Treat this as a pilot, not a final result -- the paper's methodology
  calls for expanding the QA set before this comparison is reported as a
  system-level claim.
- **What "better" means here.** The probe should be judged against the
  entropy curve as a whole, not a single point. If the probe's curve sits
  fully above and to the right of the dashed entropy line, that is real
  evidence it captures something entropy does not. If the two curves
  cross or roughly overlap, the honest conclusion is that entropy (read
  correctly, as established in Section 6.3) is already a strong baseline
  and the probe does not yet clearly improve on it.
- **Overfitting risk.** With a small test set, a single train/test split
  can be noisy. If you want a sturdier estimate, rerun this notebook a
  few times with different `random_state` values in the train/test split
  and check whether the curve shape is stable.

In [ ]:
# Save the trained probe + scaler for later use on the Pi (as a plain
# numpy-based inference function, since the Pi does not have PyTorch/GPU).
import pickle

with open("probe_and_scaler.pkl", "wb") as f:
    pickle.dump({"probe": probe, "scaler": scaler,
                "hidden_size": model.config.hidden_size}, f)

from google.colab import files
files.download("probe_and_scaler.pkl")
files.download("fig_probe_vs_entropy.pdf")
files.download("fig_probe_vs_entropy.png")
print("Downloads triggered -- check your browser's download folder.")

## Next steps

1. Bring `fig_probe_vs_entropy.png` back to the main conversation to
   interpret the result together.
2. If the probe clearly beats the entropy baseline: the next build is
   wiring a lightweight version of it into the live controller on the Pi
   (Section 5 / Algorithm 1 of the paper). Since the Pi doesn't have
   PyTorch, this typically means exporting the trained MLP's weights as
   plain matrices and reimplementing the forward pass (two matrix
   multiplies and a ReLU) in pure Python or numpy -- fast enough to run
   per boundary on the Pi without needing a GPU there.
3. If the probe does not clearly beat entropy: consider expanding the QA
   dataset (more boundary examples, especially longer/more verbose
   responses where the interesting decisions happen), or trying a richer
   feature (e.g. hidden states from a middle layer instead of only the
   last one, or averaging over the last few tokens instead of just one).
